Here we go again with Andreas corrections

In [1]:
import numpy as np
import pandas as pd
import copy as copy
import scipy.io
import time
import os
from scipy.linalg import solve, LinAlgWarning
import warnings

def highlight_greaterthan(s, threshold, column):
    is_max = pd.Series(data=False, index=s.index)
    is_max[column] = s.loc[column] >= threshold
    return ['background-color: lightgreen' if is_max.any() else '' for v in is_max]


def paso_intpoint(mu, wmu):
    eps = 0.01
    alphas = np.ones( len(mu) +1 ) # this way the last entry remains one.
    for i in range( len(mu) ):
        if wmu[i] < 0:
            alphas[i] = -(1-eps)*mu[i]/wmu[i]

    alpha = min(alphas)
    return alpha


def loadProblem( fname, useSparse = False ):
    mat = scipy.io.loadmat( fname )
    if useSparse:
        A = mat.get('Problem')[0,0][2].astype(float) # sparse matrix
    else:
        A = mat.get('Problem')[0,0][2].astype(float).toarray()
    
    b = mat.get('Problem')[0,0][3].astype(float)[:,0]
    #id = mat.get('Problem')[0,0][4]
    aux = mat.get('Problem')[0,0][5]
    c = aux[0,0][0].astype(float)[:,0]
    print(' Norma infinita de b: ', np.linalg.norm(b, np.inf) )
    
    return {'AE':A, 'bE':b, 'c':c}

Sistema Completo

In [2]:
def intpoint(Q, A, F, c, b, d):
    k = 0
    n = Q.shape[0]
    m = A.shape[0]
    p = F.shape[0]
    tol = 1e-6
    kmax = 100
    tau = 0.5

    print("El rango de A es", np.linalg.matrix_rank(A))

    AT = A.T
    FT = F.T

    # Initial values
    x = np.zeros(n)
    lamda = np.zeros(m)
    mu = np.ones(p)
    z = np.ones(p)
    e = np.ones(p)

    # Initial residuals
    v1 = Q @ x + AT @ lamda - FT @ mu + c
    v2 = A @ x - b
    v3 = -F @ x + d + z
    v4 = np.multiply(mu, z)  # Use element-wise product for complementarity condition

    ld = np.concatenate((v1, v2, v3, v4), 0)
    norma_cnpo = np.linalg.norm(ld,np.inf)

    while norma_cnpo > tol and k < kmax:
        # Update diagonal matrices Z and U inside the loop
        Z = np.diag(z)
        U = np.diag(mu)

        ### KKT Matrix
        row1 = np.hstack((Q, AT, -FT, np.zeros((n, p))))
        row2 = np.hstack((A, np.zeros((m, m + p + p))))
        row3 = np.hstack((-F, np.zeros((p, m + p)), np.identity(p)))
        row4 = np.hstack((np.zeros((p, n + m)), Z, U))

        M = np.vstack((row1, row2, row3, row4))

        # Perturb the complementarity condition (v4)
        v4_pert = np.multiply(mu, z) - tau * e
        ld_pert = np.concatenate((v1, v2, v3, v4_pert), 0)

        # Solve the linear system with the perturbed residual
        #w_vector = np.linalg.solve(M, -ld_pert)

        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            w_vector = solve(M, -ld_pert, check_finite=True)
            if w and issubclass(w[-1].category, np.linalg.LinAlgWarning):
                print("Matrix is ill-conditioned. Exiting.")
                exit()


        wx = w_vector[0:n]
        wlamda = w_vector[n:n + m]
        wmu = w_vector[n + m:n + m + p]
        wz = w_vector[n + m + p:]

        ### Step size
        alpha_mu = paso_intpoint(mu, wmu)
        alpha_z = paso_intpoint(z, wz)
        #alpha = 0.995 * min(alpha_mu, alpha_z)
        alpha = min(alpha_mu, alpha_z)

        # Update variables
        x     += alpha * wx
        mu    += alpha * wmu
        lamda += alpha * wlamda
        z     += alpha * wz

        # Update tau and residuals
        tau = np.dot(mu, z) / (2 * p)
        k += 1

        # Recalculate residuals
        v1 = Q @ x + AT @ lamda - FT @ mu + c
        v2 = A @ x - b
        v3 = -F @ x + d + z
        v4 = np.multiply(mu, z)  # Element-wise product

        ld = np.concatenate((v1, v2, v3, v4), 0)
        norma_cnpo = np.linalg.norm(ld,np.inf)

        print("iter=", k, "|", "||cnpo||=", norma_cnpo)
        print("tau =",tau)

        if norma_cnpo <= tol or k == kmax:
            return x, lamda, mu, z, k

    return x, lamda, mu, z, k

Sistema reducido

In [7]:
def intpointR(Q, A, F, c, b, d):
    k = 0
    n = Q.shape[0]
    m = A.shape[0]
    p = F.shape[0]
    
    tol = 1e-6
    kmax = 100
    tau = 0.5
    
    print("El rango de A es", np.linalg.matrix_rank(A))
    
    AT = A.T
    FT = F.T
    
    # Initial values
    x = np.ones(n)
    lamda = np.zeros(m)
    mu = np.ones(p)
    z = np.ones(p)
    #e = np.ones(p)
    
    v1 = Q @ x + AT @ lamda - FT @ mu + c
    v2 = A @ x - b
    v3 = -F @ x + d + z
    v4 = np.multiply(mu, z)  # Element-wise product
    ld1 = np.concatenate((v1, v2, v3, v4), 0)
    norma_cnpo = np.linalg.norm(ld1,np.inf)

    while norma_cnpo > tol and k < kmax:
        # Update diagonal matrices Z and U inside the loop
        # Initial residuals
        Z = np.diag(z)
        U = np.diag(mu)
        ### KKT Matrix
        row1 = np.hstack((Q, AT, -FT, np.zeros((n, p))))
        row2 = np.hstack((A, np.zeros((m, m + p + p))))
        row3 = np.hstack((-F, np.zeros((p, m + p)), np.identity(p)))
        row4 = np.hstack((np.zeros((p, n + m)), Z, U))

        M = np.vstack((row1, row2, row3, row4))
        
        D = np.diag(mu / z)
        G = G = Q+FT@D@F
        w = np.zeros((p, 1))
        for i in range(p):
            w[i] = F[i, :] @ x - d[i] - (tau / mu[i])
        w = w.ravel()
          
        dg = Q @ x + AT @ lamda - FT@mu + c + FT@D@w
        
        # Define K as a block matrix
        m = A.shape[0]
        K = np.block([
            [G, AT],
            [A, np.zeros((m, m))]
        ])
        
        # Calculate the reciprocal condition number of G
        condK = np.linalg.cond(G,1)
        
        # Define ld
        ld = -np.concatenate([dg, A @ x - b])
        norma_cnpo = np.linalg.norm(ld)
        
        # Solve the linear system
        w_vector = np.linalg.solve(K, ld)
        
        wx     = w_vector[0:n]
        wlamda = w_vector[n:n + m]
        wmu    = -D @ (F @ wx + w)
        wz     = -( (1 / mu) * (z * wmu - tau) + z )
        
        ### Step size
        alpha_mu = paso_intpoint(mu, wmu)
        alpha_z  = paso_intpoint(z, wz)
        #alpha    = 0.995 * min(alpha_mu, alpha_z)
        alpha    = min(alpha_mu, alpha_z)
        
        # remember something
        perc_mu = wmu/mu
        perc_z  = wz/z
        
        # Update variables
        x += alpha * wx
        mu += alpha * wmu
        lamda += alpha * wlamda
        z += alpha * wz
        
        # Update tau and residuals
        tau = np.dot(mu, z) / (2 * p)
        k += 1
        
        # Recalculate residuals
        v1 = Q @ x + AT @ lamda - FT @ mu + c
        v2 = A @ x - b
        v3 = -F @ x + d + z
        v4 = np.multiply(mu, z)  # Element-wise product
        
        ld1 = np.concatenate((v1, v2, v3, v4), 0)
        norma_cnpo = np.linalg.norm(ld1,np.inf)
        
        print("\niter=", k, "\t", "||cnpo||=", norma_cnpo)
        print("Condition number of G:", np.linalg.cond(G,1))
        print("rcond(G)", (1/np.linalg.cond(G,1)))
        print("tau =",tau)
        mask = mu*z <= 1e-5
        
        #print('cuantos chicos mu*z = %g, vector\n' % (sum(mask)), (mu*z)[mask])
        
        red_mu = []
        if all(mask):
            #neg_mu_mask = (-0.52 < perc_mu) & (perc_mu < -0.48)
            #const_z_mask = (-0.01 < perc_z) & (perc_z < 0.01)
            grow_z_mask = (-0.03 < perc_z) #& (perc_z < 0.01)
            neg_mu = np.arange( len(mask) )[grow_z_mask]
            
            df = pd.DataFrame({'mu': mu, 'pmu': perc_mu, 'z': z, 'pz': perc_z})
            display( df.style.apply(highlight_greaterthan, threshold=-0.02, column=['pz'], axis=1) )
            
            #print('mus chicos: vector\n', mu[neg_mu])
            if set(red_mu).issubset( neg_mu ):
                print ('IS subset: GOOD')
            else:
                print ('FAILS subset condition: BAD')
                
            #print('mus chicos: vector\n', neg_mu)
            #print('  change in percentages for mu \n', perc_mu[neg_mu] )
            #print('zs tending to positive contants\n', z[neg_mu] )
            #print('  Largest and smallest change for percentages in entries of z  \n', min(perc_z[neg_mu]), max(perc_z[neg_mu] ))
            red_mu = neg_mu.copy()
        

        if norma_cnpo <= tol or k == kmax:
            return x, lamda, mu, z, k

    return x, lamda, mu, z, k

In [8]:
set([2,3,4]).issubset([1,3,4])


False

Code running

In [9]:
"""
Load some mat-files into python.
Created on Wed Sep 29 12:42:00 2021
@author: Andreas Wachtel
"""

#mat_files = 'lp_kb2.mat'     # b = 0
#mat_files = 'lp_afiro.mat'   # bmax = 500, and no mu tends to zero  and no condition problem
mat_files = 'lp_blend.mat'   # bmax = 26.32, and no mu tends to zero.
#mat_files = 'lp_fit1d.mat'   # b = 0
#mat_files = 'lp_fit1p.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_files = 'lp_grow15.mat'  # b = 0, condition 10^6
#mat_files = 'lp_grow22.mat'  # b = 0, condition 1.4e7
#mat_files = 'lp_grow7.mat'   # b = 0, condition 1.2e7

print(mat_files)
H = loadProblem(mat_files)

# Define the quadratic matrix Q
Q = np.eye(H['c'].shape[0])

# Define the linear term vector c
c = H['c']

# Define the equality constraint matrix A and vector b
A = H['AE']
#b = H['bE']
b = np.zeros( A.shape[0] )

# Define an inequality constraint: x1 >= -10
#F = np.zeros([H['c'].shape[0],H['c'].shape[0]])
F = np.eye(H['c'].shape[0])
d = np.zeros([H['c'].shape[0],1])
d = d.ravel()  # Flattens the vector to a 1D vector

# Start the timer
start_time = time.time()

# Run the function
x, lamda, mu, z, k = intpointR(Q, A, F, c, b, d)

# Calculate elapsed time and store it in a variable
elapsed_time = time.time() - start_time

# Print the results
#print(f"Solution for x: {x}")
#print(f"Multipliers (y): {lamda}")
print(f"Number of iterations: {k}")
print(f"Elapsed time: {elapsed_time} seconds")

lp_blend.mat
 Norma infinita de b:  26.32
El rango de A es 74

iter= 1 	 ||cnpo||= 37.28696945372942
Condition number of G: 1.0
rcond(G) 1.0
tau = 0.26371114643495186

iter= 2 	 ||cnpo||= 26.623955073322055
Condition number of G: 126.80832542883512
rcond(G) 0.007885917558001351
tau = 0.21694983707193427

iter= 3 	 ||cnpo||= 9.905652042917998
Condition number of G: 4041.2055228688027
rcond(G) 0.0002474509139268206
tau = 0.11207542971695485

iter= 4 	 ||cnpo||= 3.6771079757937373
Condition number of G: 2101.423143316701
rcond(G) 0.0004758679865025604
tau = 0.05759399636186541

iter= 5 	 ||cnpo||= 1.0361806382262593
Condition number of G: 30055.75825345012
rcond(G) 3.327149465228379e-05
tau = 0.023999317541498366

iter= 6 	 ||cnpo||= 0.2048028143464283
Condition number of G: 45377.31665042313
rcond(G) 2.2037442357021245e-05
tau = 0.00774435022192755

iter= 7 	 ||cnpo||= 0.05212740118414182
Condition number of G: 898300.3147570044
rcond(G) 1.1132134583193438e-06
tau = 0.0030707863635410283

,mu,pmu,z,pz
0,257.746441,0.655283,0.000000,-1.017747
1,254.929920,0.655554,0.000000,-0.983962
2,477.382863,0.655501,0.000000,-1.041008
3,34.401566,0.654030,0.000000,-0.687753
4,315.746957,0.656841,0.000000,-0.982137
5,401.146117,0.655861,0.000000,-1.006528
6,296.241042,0.655350,0.000000,-1.002066
7,357.601529,0.655803,0.000000,-1.011494
8,407.964470,0.655488,0.000000,-1.104808
9,37.662446,0.654360,0.000000,-0.448129


IS subset: GOOD

iter= 17 	 ||cnpo||= 1.4763390737445948e-06
Condition number of G: 103242864649.17606
rcond(G) 9.685899392641277e-12
tau = 2.0547757466941628e-07


,mu,pmu,z,pz
0,381.632439,0.653429,0.000000,-0.976958
1,377.493723,0.653598,0.000000,-1.018527
2,706.906469,0.653626,0.000000,-0.941931
3,50.917642,0.652676,0.000000,-1.184418
4,467.719731,0.654328,0.000000,-1.020457
5,594.091408,0.653884,0.000000,-0.991904
6,438.669297,0.653612,0.000000,-0.997583
7,529.608241,0.653906,0.000000,-0.985514
8,604.132588,0.653695,0.000000,-0.801755
9,55.751965,0.652961,0.000000,-1.233924


IS subset: GOOD

iter= 18 	 ||cnpo||= 4.915043697758344e-07
Condition number of G: 49129827115.86542
rcond(G) 2.0354234050969652e-11
tau = 6.941646396262448e-08


,mu,pmu,z,pz
0,572.052088,0.643323,0.000000,-1.017451
1,565.882047,0.643439,0.000000,-0.984287
2,1059.679333,0.643422,0.000000,-1.040258
3,76.301788,0.642772,0.000000,-0.695740
4,701.340858,0.644005,0.000000,-0.982593
5,890.633688,0.643571,0.000000,-1.006346
6,657.557774,0.643352,0.000000,-1.001859
7,793.954267,0.643548,0.000000,-1.011152
8,905.614159,0.643415,0.000000,-1.102885
9,83.552411,0.642916,0.000000,-0.461407


IS subset: GOOD
Number of iterations: 18
Elapsed time: 0.1147310733795166 seconds


In [154]:
zv = 1.13331097e-01
zn =  1.13304722e-01
(zn-zv)/zv
mun = 6.39708019e-07
muv = 1.27047154e-06
(muv-mun)/muv

0.4964798510952871